# 라이브러리 임포트

환경 변수와 Supabase Python 클라이언트를 사용하기 위한 라이브러리를 불러옵니다.

In [1]:
import os  # 운영체제의 환경 변수에 접근하는 표준 라이브러리
from dotenv import load_dotenv  # .env 파일을 읽어 환경 변수로 등록하는 함수
from supabase import create_client, Client  # 클라이언트 생성 함수와 타입

# .env 파일을 환경 변수로 등록

API 키를 코드에 직접 작성하지 않고 `.env` 파일에서 안전하게 불러옵니다.

In [2]:
# 현재 폴더의 .env 파일을 찾아 키와 값을 환경 변수로 등록합니다.
load_dotenv()  # 파일을 찾고 읽었다면 True를 반환합니다.

True

# 환경 변수 값을 Python 변수에 저장

`os.environ[키]`는 필수 환경 변수를 읽으며, 키가 없으면 `KeyError`가 발생합니다.

> **주의:** Service Role Key는 관리자 권한을 가지므로 공개 저장소나 브라우저 코드에 노출하면 안 됩니다.

In [3]:
# Supabase 프로젝트 주소와 서버 전용 API 키를 환경 변수에서 읽습니다.
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

# Supabase 클라이언트 생성

이후의 모든 테이블 조회는 생성한 `supabase` 객체를 통해 실행합니다.

In [4]:
# : Client는 변수의 예상 타입을 나타내는 타입 힌트입니다.
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

# 간단한 조회로 연결 테스트

`app_users` 테이블에서 최대 한 행을 조회해 Supabase 연결 상태를 확인합니다.

In [5]:
result = (
    supabase.table("app_users")  # SQL의 FROM app_users
    .select("*")  # SQL의 SELECT *
    .limit(1)  # SQL의 LIMIT 1
    .execute()  # 지금까지 만든 쿼리를 Supabase에 전송해 실제로 실행
)
print(result.data)  # [] 또는 데이터 리스트 출력되면 연결 성공

[{'id': '4c460602-eaf4-4e6e-a1d2-781843690bf7', 'username': 'kim', 'display_name': '김학생', 'created_at': '2026-06-17T11:24:57.807583+00:00'}]


# 데이터 조회

Supabase Python 클라이언트는 SQL을 메서드 체이닝으로 표현합니다. 마지막에 `execute()`를 호출해야 쿼리가 실행되고, 조회된 행 목록은 `result.data`에 저장됩니다.

### 전체 열과 행 조회

`.select("*")`는 SQL의 `SELECT *`와 같으며, 결과가 없으면 빈 리스트 `[]`를 반환합니다.

In [6]:
# SELECT * FROM app_users에 해당하는 쿼리입니다.
result = supabase.table("app_users").select("*").execute()
users = result.data  # 각 행이 딕셔너리인 리스트
print(users)

[{'id': '4c460602-eaf4-4e6e-a1d2-781843690bf7', 'username': 'kim', 'display_name': '김학생', 'created_at': '2026-06-17T11:24:57.807583+00:00'}, {'id': '11acf377-3c31-42c9-a5fd-6cd9c0d8b4a2', 'username': 'lee', 'display_name': '이학생', 'created_at': '2026-06-17T11:36:08.387918+00:00'}]


In [8]:
result = (
    supabase.table("app_users")  # 조회할 테이블 지정
    .select("id, username, created_at")  # 필요한 열만 선택
    .order("created_at", desc=True)  # created_at 기준 내림차순(DESC) 정렬
    .execute()  # 쿼리 실행
)

result.data  # 주피터에서는 마지막 표현식의 값을 표 형태로 출력

[{'id': '11acf377-3c31-42c9-a5fd-6cd9c0d8b4a2',
  'username': 'lee',
  'created_at': '2026-06-17T11:36:08.387918+00:00'},
 {'id': '4c460602-eaf4-4e6e-a1d2-781843690bf7',
  'username': 'kim',
  'created_at': '2026-06-17T11:24:57.807583+00:00'}]

In [10]:
result = (
    supabase.table("conversations")  # conversations 테이블 선택
    .select("*")  # 모든 열 조회
    .eq("user_id", "11acf377-3c31-42c9-a5fd-6cd9c0d8b4a2")  # SQL의 WHERE user_id = '...'
    .order("created_at", desc=True)  # 최신 대화부터 정렬
    .execute()  # 조건과 정렬을 포함한 쿼리 실행
)

conversations = result.data  # 조건에 맞는 행 목록을 변수에 저장
conversations  # 결과가 없으면 [] 출력

[]